<a href="https://colab.research.google.com/github/tryan01/gravitational-waves/blob/main/microquant_v1_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import pandas as pd
import numpy as np
import gcsfs
from google.colab import auth
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict

# 1. Authenticate Colab
print("Authenticating with Google Cloud...")
auth.authenticate_user()

# 2. Connect to Bucket
fs = gcsfs.GCSFileSystem()
bucket_path = 'tryanheider-kalshi-bucket/kalshi_dense_state_v1'

print(f"Scanning bucket: gs://{bucket_path}...")
all_dense_files = fs.glob(f"{bucket_path}/*.parquet")
all_dense_files = [f"gs://{f}" for f in all_dense_files]

if not all_dense_files:
    raise FileNotFoundError("No parquet files found in the bucket. Check your path.")

# 3. Pair Markets to Prevent Leakage
market_groups = {}
for file_path in all_dense_files:
    # Example filename: dense_KXMLBGAME-26APR151840KCDET-KC.parquet
    filename = os.path.basename(file_path)
    base_game = filename.split('-')[1] # Extracts '26APR151840KCDET'

    if base_game not in market_groups:
        market_groups[base_game] = []
    market_groups[base_game].append(file_path)

unique_games = list(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Split 80/20 strictly by GAME
train_games, val_games = train_test_split(unique_games, test_size=0.2, random_state=42)

train_files = [f for game in train_games for f in market_groups[game]]
val_files = [f for game in val_games for f in market_groups[game]]

print(f"Train Set: {len(train_games)} games ({len(train_files)} individual tickers)")
print(f"Val Set: {len(val_games)} games ({len(val_files)} individual tickers)")

Authenticating with Google Cloud...
Scanning bucket: gs://tryanheider-kalshi-bucket/kalshi_dense_state_v1...
Found 22 unique MLB games.
Train Set: 17 games (34 individual tickers)
Val Set: 5 games (10 individual tickers)


In [35]:
class KalshiDenseDataset(Dataset):
    def __init__(self, file_paths: List[str], seq_len: int = 60, forward_look: int = 30, threshold: float = 0.015, stride: int = 5):
        self.seq_len = seq_len
        self.forward_look = forward_look
        self.threshold = threshold
        self.stride = stride

        self.tensors = []
        self.labels = []
        self.timestamps = []
        self.audit_log = {'class_0': 0, 'class_1': 0, 'class_2': 0, 'nan_drops': 0}

        # Explicitly define columns that require log1p normalization
        self.vol_cols = [f'bid_vol_{i}' for i in range(1, 6)] + \
                        [f'ask_vol_{i}' for i in range(1, 6)] + \
                        ['taker_vol_yes', 'taker_vol_no']

        print(f"Loading {len(file_paths)} dense files into RAM...")
        for f in file_paths:
            self._process_file(f)

        print("\n=== FINAL ENGINEERING AUDIT ===")
        print(f"Total valid {seq_len}-step sequences: {len(self.tensors)}")
        print(f"Distribution -> DOWN: {self.audit_log['class_0']} | FLAT: {self.audit_log['class_1']} | UP: {self.audit_log['class_2']}")
        print(f"Dropped due to NaNs: {self.audit_log['nan_drops']}")

    def _process_file(self, file_path: str):
        df = pd.read_parquet(file_path)
        if df.empty:
            return

        # 1. Target Math (Must be done BEFORE log normalization)
        # Adding 1e-8 prevents division by zero if book momentarily empties
        df['micro_price'] = (df['ask_vol_1'] * df['bid_px_1'] + df['bid_vol_1'] * df['ask_px_1']) / (df['bid_vol_1'] + df['ask_vol_1'] + 1e-8)
        df['smoothed_target'] = df['micro_price'].rolling(10).mean().shift(-self.forward_look)

        # 2. Log1p Normalization
        for col in self.vol_cols:
            df[col] = np.log1p(df[col].values)

        # 3. Define Feature Vector (Drop target metrics and pure ts)
        feature_cols = [c for c in df.columns if c not in ['ts', 'micro_price', 'smoothed_target']]

        # 4. Extract Sliding Windows
        for i in range(0, len(df) - self.seq_len - self.forward_look, self.stride):
            window = df.iloc[i : i + self.seq_len][feature_cols].values

            # Labeling is strictly based on the final step of the window vs the future
            current_micro = df.iloc[i + self.seq_len - 1]['micro_price']
            future_micro = df.iloc[i + self.seq_len - 1]['smoothed_target']
            target_px_move = future_micro - current_micro

            trigger_ts = df.iloc[i + self.seq_len - 1]['ts']

            if np.isnan(window).any() or np.isnan(target_px_move):
                self.audit_log['nan_drops'] += 1
                continue

            # Classify: 0 (Down), 1 (Flat), 2 (Up)
            if target_px_move > self.threshold:
                label = 2
                self.audit_log['class_2'] += 1
            elif target_px_move < -self.threshold:
                label = 0
                self.audit_log['class_0'] += 1
            else:
                label = 1
                self.audit_log['class_1'] += 1

            self.tensors.append(torch.tensor(window, dtype=torch.float32))
            self.labels.append(label)
            self.timestamps.append(trigger_ts)

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx], self.timestamps[idx]

# Instantiate DataLoaders
print("\nBuilding Training Dataset...")
train_dataset = KalshiDenseDataset(train_files, seq_len=60, forward_look=30, threshold=0.015, stride=5)

print("\nBuilding Validation Dataset...")
val_dataset = KalshiDenseDataset(val_files, seq_len=60, forward_look=30, threshold=0.015, stride=5)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"\nDataLoader Ready. Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Building Training Dataset...
Loading 34 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 97308
Distribution -> DOWN: 6958 | FLAT: 83425 | UP: 6925
Dropped due to NaNs: 0

Building Validation Dataset...
Loading 10 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 28620
Distribution -> DOWN: 1645 | FLAT: 25287 | UP: 1688
Dropped due to NaNs: 0

DataLoader Ready. Train batches: 1521 | Val batches: 448


In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Shape: (1, max_len, d_model) for batch_first=True
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model

        # 1. Input Projection: Map 23 raw features to 64-dim embedding
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder (batch_first=True ensures shape is [batch, seq, feature])
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, feature_dim]
        Returns:
            logits: Tensor, shape [batch_size, num_classes]
        """
        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation for sequence classification
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits



class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        """
        Args:
            inputs: [batch_size, num_classes] (raw logits)
            targets: [batch_size] (integer class indices)
        """
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)

        # Gather the alpha weights for the target classes
        alpha_t: Tensor = self.alpha.gather(0, targets)

        # Apply the focal equation
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [4]:
from tqdm import tqdm

# --- Setup Devices and Hyperparameters ---
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model: nn.Module = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2).to(device)
optimizer: torch.optim.Optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Calculate empirical Alpha weights based on your Train distribution
# Inverse frequency weighting: total_samples / (num_classes * class_samples)
class_counts = torch.tensor([5084, 87474, 4750], dtype=torch.float32)
alpha_weights: Tensor = class_counts.sum() / (3.0 * class_counts)
alpha_weights = alpha_weights.to(device)

criterion: nn.Module = FocalLoss(alpha=alpha_weights, gamma=2.0)

# --- Training Loop ---
epochs: int = 10

for epoch in range(epochs):
    model.train()
    running_loss: float = 0.0
    correct: int = 0
    total: int = 0

    # Progress bar wrapping the training loader
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for X_batch, Y_batch in train_bar:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer.zero_grad()
        logits: Tensor = model(X_batch)
        loss: Tensor = criterion(logits, Y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Prevent explosions
        optimizer.step()

        running_loss += loss.item()

        # Compute basic accuracy
        preds: Tensor = torch.argmax(logits, dim=1)
        correct += (preds == Y_batch).sum().item()
        total += Y_batch.size(0)

        # Update progress bar metrics
        train_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

    train_acc: float = correct / total

    # --- Validation ---
    model.eval()
    val_loss: float = 0.0
    val_correct: int = 0
    val_total: int = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val  ]")
        for X_batch, Y_batch in val_bar:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            logits: Tensor = model(X_batch)
            loss: Tensor = criterion(logits, Y_batch)

            val_loss += loss.item()
            preds: Tensor = torch.argmax(logits, dim=1)
            val_correct += (preds == Y_batch).sum().item()
            val_total += Y_batch.size(0)

    val_acc: float = val_correct / val_total

    print(f"Epoch {epoch+1} Summary -> Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}\n")

Using device: cuda


Epoch 1/10 [Val  ]: 100%|██████████| 448/448 [00:01<00:00, 275.28it/s]


Epoch 1 Summary -> Train Loss: 0.4934 | Train Acc: 0.5483 || Val Loss: 0.4055 | Val Acc: 0.5833



Epoch 2/10 [Val  ]: 100%|██████████| 448/448 [00:01<00:00, 280.68it/s]


Epoch 2 Summary -> Train Loss: 0.4749 | Train Acc: 0.5614 || Val Loss: 0.3990 | Val Acc: 0.6134



...

Epoch 8 Summary -> Train Loss: 0.4496 | Train Acc: 0.5830 || Val Loss: 0.3986 | Val Acc: 0.5875



Epoch 9/10 [Val  ]: 100%|██████████| 448/448 [00:01<00:00, 273.26it/s]


Epoch 9 Summary -> Train Loss: 0.4447 | Train Acc: 0.5887 || Val Loss: 0.4143 | Val Acc: 0.6334



Epoch 10/10 [Val  ]: 100%|██████████| 448/448 [00:01<00:00, 273.51it/s]

Epoch 10 Summary -> Train Loss: 0.4404 | Train Acc: 0.5901 || Val Loss: 0.4131 | Val Acc: 0.6378



Evaluation suite:
Below are our evaluation metrics and data audits:

In [4]:
#class KalshiBacktestSimulator:
    def __init__(self, latency_penalty_ms: int = 50, ttl_seconds: int = 15):
        self.latency = latency_penalty_ms / 1000.0
        self.ttl = ttl_seconds
        self.audit = {'trades_fired': 0, 'filled': 0, 'cancelled_ttl': 0, 'stopped_out': 0, 'gross_pnl': 0.0}

    def evaluate_model_trigger(self, market_df: pd.DataFrame, trigger_ts: float, side_to_join: str, target_price: float):
        """
        Simulates worst-case queue position for a triggered model signal.
        market_df: The raw chronological event log for the specific market.
        trigger_ts: The exact local_recv_ts when the model generated the tensor.
        """
        self.audit['trades_fired'] += 1

        # 1. Enforce Execution Latency
        arrival_time = trigger_ts + self.latency

        # 2. Slice future reality
        future_df = market_df[market_df['local_recv_ts'] >= arrival_time].copy()
        if future_df.empty:
            return 0.0 # Market closed or no future data

        # 3. Snapshot Queue Ahead
        queue_ahead = self._get_resting_volume_at_arrival(future_df, target_price, side_to_join)

        # 4. Walk forward in time to simulate fill or TTL kill
        for _, event in future_df.iterrows():
            current_time = event['local_recv_ts']

            # Kill Switch 1: Time-To-Live expiration
            if current_time - arrival_time > self.ttl:
                self.audit['cancelled_ttl'] += 1
                return 0.0

            # Monitor Taker Flow to deplete queue
            if event['event_type'] == 'trade':
                # Map taker side to check if it's eating our queue
                trade_price = event['price_dollars'] if event['side'] == 'yes' else round(1.0 - event['price_dollars'], 2)

                if trade_price == target_price:
                    queue_ahead -= event['count_fp']

                    # Fill Condition!
                    if queue_ahead <= 0:
                        self.audit['filled'] += 1
                        return self._manage_inventory(future_df, current_time, side_to_join, target_price)

        return 0.0

    def _get_resting_volume_at_arrival(self, future_df, price, side):
        # In full implementation, this uses the exact LOB state at arrival microsecond
        # Placeholder assuming 500 contracts ahead for baseline test
        return 500.0

    def _manage_inventory(self, future_df, fill_time, side, entry_price):
        # Post-Fill Logic: Rest an order on the opposite side to capture the spread
        # Placeholder for returning a standardized successful spread capture PnL (+0.01)
        self.audit['gross_pnl'] += 0.01
        return 0.01

    def print_autopsy(self):
        print("\n=== PnL Simulator Autopsy ===")
        print(f"Total Signals Fired: {self.audit['trades_fired']}")
        print(f"Orders Successfully Filled: {self.audit['filled']}")
        print(f"Orders Cancelled (TTL/Adverse): {self.audit['cancelled_ttl']}")
        print(f"Gross PnL (Maker/Maker): +${self.audit['gross_pnl']:.2f}")

# Initialize simulator for testing later
simulator = KalshiBacktestSimulator()

In [17]:
# --- Visual Tensor Audit ---
# Pop exactly one batch of 64 sequences from the Train Loader
data_iter = iter(train_loader)
X_batch, Y_batch = next(data_iter)

print("=== TENSOR SHAPE AUDIT ===")
print(f"X_batch shape: {X_batch.shape} -> (Batch Size, Sequence Length, Features)")
print(f"Y_batch shape: {Y_batch.shape} -> (Batch Size)\n")

print("=== VISUAL STATE AUDIT (First Sequence, Last Time Step) ===")
# Look at the very last 1-second step of the 60-second window for the first tensor
final_step_features = X_batch[0][-1]

feature_names = [
    'bid_px_1', 'bid_vol_1', 'ask_px_1', 'ask_vol_1',
    'bid_px_2', 'bid_vol_2', 'ask_px_2', 'ask_vol_2',
    'bid_px_3', 'bid_vol_3', 'ask_px_3', 'ask_vol_3',
    'time_fraction'
]

print("State Matrix:")
for name, val in zip(feature_names, final_step_features):
    print(f"{name:>13}: {val.item():.4f}")

print(f"\nTarget Label: {Y_batch[0].item()} (0=DOWN, 1=FLAT, 2=UP)")

=== TENSOR SHAPE AUDIT ===
X_batch shape: torch.Size([64, 60, 23]) -> (Batch Size, Sequence Length, Features)
Y_batch shape: torch.Size([64]) -> (Batch Size)

=== VISUAL STATE AUDIT (First Sequence, Last Time Step) ===
State Matrix:
     bid_px_1: 0.5900
    bid_vol_1: 11.2226
     ask_px_1: 0.6000
    ask_vol_1: 13.0523
     bid_px_2: 0.5800
    bid_vol_2: 13.3472
     ask_px_2: 0.6100
    ask_vol_2: 13.8169
     bid_px_3: 0.5700
    bid_vol_3: 12.1534
     ask_px_3: 0.6200
    ask_vol_3: 13.3552
time_fraction: 0.5600

Target Label: 1 (0=DOWN, 1=FLAT, 2=UP)


In [5]:
import numpy as np
import torch.nn.functional as F
from typing import Dict

# 1. Set model to evaluation mode
model.eval()

all_preds_list: list[np.ndarray] = []
all_probs_list: list[np.ndarray] = []
all_targets_list: list[np.ndarray] = []

print("Extracting validation predictions for simulator...")

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        # Get raw logits
        logits: torch.Tensor = model(X_batch)

        # Convert to rigorous probabilities [0.0 to 1.0]
        probs: torch.Tensor = F.softmax(logits, dim=1)
        preds: torch.Tensor = torch.argmax(probs, dim=1)

        all_preds_list.append(preds.cpu().numpy())
        all_probs_list.append(probs.cpu().numpy())
        all_targets_list.append(Y_batch.cpu().numpy())

# 2. Concatenate into flat, contiguous arrays
all_preds: np.ndarray = np.concatenate(all_preds_list, axis=0)
all_probs: np.ndarray = np.concatenate(all_probs_list, axis=0)
all_targets: np.ndarray = np.concatenate(all_targets_list, axis=0)

# 3. Diagnostic Audit
unique, counts = np.unique(all_preds, return_counts=True)
pred_dist: Dict[int, int] = dict(zip(unique, counts))

print("\n=== VALIDATION PREDICTION DISTRIBUTION ===")
print(f"Total Sequences: {len(all_preds)}")
print(f"Predicted DOWN (0): {pred_dist.get(0, 0)}")
print(f"Predicted FLAT (1): {pred_dist.get(1, 0)}")
print(f"Predicted UP   (2): {pred_dist.get(2, 0)}")

Extracting validation predictions for simulator...

=== VALIDATION PREDICTION DISTRIBUTION ===
Total Sequences: 28620
Predicted DOWN (0): 4360
Predicted FLAT (1): 16831
Predicted UP   (2): 7429


In [6]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from typing import List

# These arrays were generated in the previous extraction block
# all_preds: np.ndarray
# all_targets: np.ndarray

target_names: List[str] = ['DOWN (0)', 'FLAT (1)', 'UP (2)']

print("=== RIGOROUS CLASSIFICATION METRICS ===")
report: str = classification_report(all_targets, all_preds, target_names=target_names, digits=4)
print(report)

print("\n=== CONFUSION MATRIX ===")
cm: np.ndarray = confusion_matrix(all_targets, all_preds)

print("Rows = True Class | Columns = Predicted Class [DOWN, FLAT, UP]")
print("-" * 65)
print(f"True DOWN (0) -> {cm[0]}")
print(f"True FLAT (1) -> {cm[1]}")
print(f"True UP   (2) -> {cm[2]}")

=== RIGOROUS CLASSIFICATION METRICS ===
              precision    recall  f1-score   support

    DOWN (0)     0.1821    0.4827    0.2644      1645
    FLAT (1)     0.9774    0.6505    0.7811     25287
      UP (2)     0.1360    0.5983    0.2216      1688

    accuracy                         0.6378     28620
   macro avg     0.4318    0.5772    0.4224     28620
weighted avg     0.8820    0.6378    0.7184     28620


=== CONFUSION MATRIX ===
Rows = True Class | Columns = Predicted Class [DOWN, FLAT, UP]
-----------------------------------------------------------------
True DOWN (0) -> [794 182 669]
True FLAT (1) -> [ 3087 16450  5750]
True UP   (2) -> [ 479  199 1010]


In [12]:
import numpy as np
from typing import Dict, List

# Configuration
CONFIDENCE_THRESHOLD: float = 0.6
FEE_MAKER: float = 0.0000  # Kalshi Maker fee is usually zero or rebate
SPREAD_PENALTY: float = 0.01 # Assume a standard 1-cent spread cost if forced to cross

trades_triggered: int = 0
gross_pnl: float = 0.0
wins: int = 0
losses: int = 0

print(f"--- Running Maker EV Simulation (Threshold: {CONFIDENCE_THRESHOLD}) ---")

# We iterate through the exact length of the validation arrays
for i in range(len(all_preds)):
    pred_class: int = all_preds[i]
    confidence: float = all_probs[i, pred_class]

    # Ignore FLAT predictions and low-confidence signals
    if pred_class == 1 or confidence < CONFIDENCE_THRESHOLD:
        continue

    # Extract the final step of the sequence from the dataset tensor
    # Feature Indices (based on our dataset class):
    # 0 = bid_px_1, 2 = ask_px_1
    current_state: torch.Tensor = val_dataset.tensors[i][-1]
    bid_px: float = current_state[0].item()
    ask_px: float = current_state[2].item()

    # We use the true future label to simulate our directional exit
    true_class: int = all_targets[i]

    # ---------------------------------------------------------
    # Maker Execution Logic: Post a passive limit order
    # ---------------------------------------------------------
    if pred_class == 2: # Model predicts UP
        # We post a Bid at current bid_px to get filled passively
        trades_triggered += 1
        if true_class == 2: # Market actually went UP
            # We got filled at bid_px, and scratched/exited at the new higher bid
            gross_pnl += (ask_px - bid_px) - FEE_MAKER
            wins += 1
        elif true_class == 0: # Market went DOWN (False Positive)
            # We caught a falling knife. Forced to exit at the new lower bid.
            gross_pnl -= SPREAD_PENALTY
            losses += 1

    elif pred_class == 0: # Model predicts DOWN
        # We post an Ask at current ask_px
        trades_triggered += 1
        if true_class == 0: # Market actually went DOWN
            gross_pnl += (ask_px - bid_px) - FEE_MAKER
            wins += 1
        elif true_class == 2: # Market went UP (False Positive)
            gross_pnl -= SPREAD_PENALTY
            losses += 1

print(f"Total Trades Simulated: {trades_triggered}")
print(f"Wins: {wins} | Losses: {losses}")
if trades_triggered > 0:
    print(f"Win Rate: {(wins/trades_triggered)*100:.2f}%")
print(f"Estimated Gross PnL (cents): {gross_pnl * 100:.2f}¢")

--- Running Maker EV Simulation (Threshold: 0.6) ---
Total Trades Simulated: 518
Wins: 208 | Losses: 58
Win Rate: 40.15%
Estimated Gross PnL (cents): 585.00¢


In [13]:
import numpy as np

print("--- INVESTIGATING HIGH CONFIDENCE TRADES (>0.7) ---")
threshold = 0.75

# Find indices where confidence > 0.8 and it didn't predict FLAT
high_conf_indices = []
for i in range(len(all_preds)):
    pred_class = all_preds[i]
    confidence = all_probs[i, pred_class]
    if pred_class != 1 and confidence >= threshold:
        high_conf_indices.append(i)

print(f"Found {len(high_conf_indices)} trades. Dumping state matrices...\n")

for idx in high_conf_indices:
    pred_class = all_preds[idx]
    true_class = all_targets[idx]
    confidence = all_probs[idx, pred_class]

    # State at prediction time (last step of the 60s window)
    current_state = val_dataset.tensors[idx][-1].numpy()

    # State 30 seconds later (stride is 5, so 30s lookahead = 6 windows forward)
    # Using min() to prevent indexing errors if it happens at the very end of the array
    future_idx = min(idx + 6, len(val_dataset) - 1)
    future_state = val_dataset.tensors[future_idx][-1].numpy()

    # Feature map: 0=bid_px_1, 1=bid_vol_1, 2=ask_px_1, 3=ask_vol_1
    bid_px_curr = current_state[0]
    # np.expm1 reverses the np.log1p scaling we applied
    bid_vol_curr = np.expm1(current_state[1])
    ask_px_curr = current_state[2]
    ask_vol_curr = np.expm1(current_state[3])

    bid_px_fut = future_state[0]
    ask_px_fut = future_state[2]

    direction = "UP" if pred_class == 2 else "DOWN"
    true_dir = "UP" if true_class == 2 else ("DOWN" if true_class == 0 else "FLAT")

    print(f"Index: {idx} | MODEL PREDICTED: {direction} ({confidence*100:.1f}%) | ACTUAL TARGET: {true_dir}")
    print(f"  [Time T]      Bid: {bid_px_curr:.2f} (Vol: {bid_vol_curr:.0f})  ||  Ask: {ask_px_curr:.2f} (Vol: {ask_vol_curr:.0f})")
    print(f"  [Time T+30s]  Bid: {bid_px_fut:.2f}               ||  Ask: {ask_px_fut:.2f}")
    print(f"  Spread Width at T: {(ask_px_curr - bid_px_curr):.2f}")
    print("-" * 65)

--- INVESTIGATING HIGH CONFIDENCE TRADES (>0.7) ---
Found 0 trades. Dumping state matrices...



In [14]:
import numpy as np

print("=== CONFIDENCE THRESHOLD AUTOPSY ===")
print(f"{'Threshold':<10} | {'Signals Fired':<15} | {'True UP/DOWN':<15} | {'Precision (Hit Rate)':<20}")
print("-" * 65)

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

for thresh in thresholds:
    # Find all indices where model predicted a tail event (0=DOWN, 2=UP) WITH confidence >= threshold
    tail_mask = ((all_preds == 0) | (all_preds == 2)) & (all_probs.max(axis=1) >= thresh)

    signals_fired = np.sum(tail_mask)

    if signals_fired == 0:
        print(f"{thresh:<10.2f} | {signals_fired:<15} | {'0':<15} | {'N/A':<20}")
        continue

    # Get the actual targets for these specific high-confidence signals
    true_classes = all_targets[tail_mask]
    pred_classes = all_preds[tail_mask]

    # A "Hit" is when the model predicted UP and it went UP, or predicted DOWN and it went DOWN
    hits = np.sum(true_classes == pred_classes)
    precision = hits / signals_fired

    print(f"{thresh:<10.2f} | {signals_fired:<15} | {hits:<15} | {precision*100:.2f}%")

print("-" * 65)

=== CONFIDENCE THRESHOLD AUTOPSY ===
Threshold  | Signals Fired   | True UP/DOWN    | Precision (Hit Rate)
-----------------------------------------------------------------
0.50       | 2168            | 671             | 30.95%
0.60       | 518             | 208             | 40.15%
0.70       | 21              | 8               | 38.10%
0.80       | 0               | 0               | N/A                 
0.90       | 0               | 0               | N/A                 
-----------------------------------------------------------------


In [36]:
import pandas as pd
import numpy as np

class KalshiBacktestSimulator:
    def __init__(self, latency_penalty_ms: int = 50, ttl_seconds: int = 15):
        self.latency = latency_penalty_ms / 1000.0
        self.ttl = ttl_seconds
        self.audit = {'trades_fired': 0, 'filled': 0, 'cancelled_ttl': 0, 'stopped_out': 0, 'gross_pnl': 0.0}

    def run_market(self, predictions: np.ndarray, probabilities: np.ndarray, states: np.ndarray, timestamps: np.ndarray, raw_market_df: pd.DataFrame, threshold: float):
        """
        The bridge between the Neural Network arrays and the raw chronological simulator.
        """
        game_pnl = 0.0
        game_trades = 0

        for i in range(len(predictions)):
            pred_class = predictions[i]
            confidence = probabilities[i, pred_class]

            # Skip FLAT predictions or low confidence
            if pred_class == 1 or confidence < threshold:
                continue

            trigger_ts = timestamps[i]

            # State mapping from our 15-feature dense tensor (assumes standard layout)
            # 0=bid_px_1, 1=bid_vol_1, 6=ask_px_1, 7=ask_vol_1
            current_state = states[i][-1]

            # 0 = DOWN (We post an Ask at the current Ask price)
            # 2 = UP (We post a Bid at the current Bid price)
            if pred_class == 0:
                side_to_join = 'no' # Functionally placing an Ask
                target_price = current_state[2] # ask_px_1
                queue_ahead_at_t = np.expm1(current_state[3]) # Reversed log1p ask_vol_1
            else:
                side_to_join = 'yes' # Placing a Bid
                target_price = current_state[0] # bid_px_1
                queue_ahead_at_t = np.expm1(current_state[1]) # Reversed log1p bid_vol_1

            # Send to the execution loop
            pnl = self.evaluate_model_trigger(
                market_df=raw_market_df,
                trigger_ts=trigger_ts,
                side_to_join=side_to_join,
                target_price=target_price,
                assumed_queue=queue_ahead_at_t
            )

            game_pnl += pnl
            if pnl != 0.0: # If it wasn't cancelled by TTL
                game_trades += 1

        return game_pnl, game_trades

    def evaluate_model_trigger(self, market_df: pd.DataFrame, trigger_ts: float, side_to_join: str, target_price: float, assumed_queue: float):
        self.audit['trades_fired'] += 1

        arrival_time = trigger_ts + self.latency

        # 1. Slice future reality from the RAW event log
        future_df = market_df[market_df['local_recv_ts'] >= arrival_time]
        if future_df.empty:
            return 0.0

        queue_ahead = assumed_queue

        # 2. Walk forward in raw tick time
        for _, event in future_df.iterrows():
            current_time = event['local_recv_ts']

            if current_time - arrival_time > self.ttl:
                self.audit['cancelled_ttl'] += 1
                return 0.0

            # Monitor Taker Flow to deplete queue
            if event['event_type'] == 'trade':
                # FIX 1: Trades use 'yes_price_dollars' and 'taker_side'
                yes_px = event.get('yes_price_dollars')

                if yes_px is not None and not pd.isna(yes_px):
                    trade_price = float(yes_px)
                else:
                    # Fallback just in case Kalshi schema maps to price_dollars
                    raw_px = event.get('price_dollars')
                    if raw_px is None or pd.isna(raw_px):
                        continue  # Malformed row, skip safely
                    trade_price = float(raw_px) if event.get('taker_side', 'yes') == 'yes' else round(1.0 - float(raw_px), 2)

                trade_side = event.get('taker_side', 'yes')

                # If taker flow hits our price level on our side
                if trade_price == target_price and trade_side != side_to_join:
                    # FIX 2: Kalshi trades log volume under 'count_fp' or 'count'
                    trade_vol = float(event.get('count_fp', event.get('count', 0.0)))
                    queue_ahead -= trade_vol

                    if queue_ahead <= 0:
                        self.audit['filled'] += 1
                        return self._manage_inventory()

        return 0.0

    def _manage_inventory(self):
        # We assume a fixed +1 cent capture if filled for this V0.1 test.
        # In V0.2, this will trigger a secondary loop to track the exit order.
        profit = 0.01
        self.audit['gross_pnl'] += profit
        return profit

    def print_autopsy(self):
        print("\n=== PnL Simulator Autopsy ===")
        print(f"Total Signals Fired: {self.audit['trades_fired']}")
        print(f"Orders Successfully Filled: {self.audit['filled']}")
        print(f"Orders Cancelled (TTL/Adverse): {self.audit['cancelled_ttl']}")
        print(f"Gross PnL (Maker/Maker): +${self.audit['gross_pnl']:.2f}")

In [18]:
import torch
import numpy as np
import torch.nn.functional as F
from typing import List

def evaluate_model_with_simulator(alpha_model: torch.nn.Module, val_file_paths: List[str], simulator_instance, trigger_threshold: float = 0.70):
    """
    Plug-and-play wrapper to test any Alpha Model against the KalshiBacktestSimulator.
    """
    alpha_model.eval()
    print(f"--- Booting Simulator Harness on {len(val_file_paths)} Validation Markets ---")

    total_pnl = 0.0
    total_trades = 0

    for file_path in val_file_paths:
        # 1. Create a strict, UN-SHUFFLED, continuous dataset for THIS SINGLE MARKET
        # Setting stride=1 gives the simulator a perfect second-by-second timeline
        market_dataset = KalshiDenseDataset([file_path], seq_len=60, forward_look=30, threshold=0.015, stride=1)

        if len(market_dataset) == 0:
            continue

        market_loader = DataLoader(market_dataset, batch_size=256, shuffle=False)

        all_preds_game, all_probs_game, all_states_game = [], [], []

        # 2. Extract predictions sequentially
        with torch.no_grad():
            for X_batch, _ in market_loader:
                X_batch = X_batch.to(device)

                logits = alpha_model(X_batch)
                probs = F.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_preds_game.append(preds.cpu().numpy())
                all_probs_game.append(probs.cpu().numpy())
                all_states_game.append(X_batch.cpu().numpy())

        # Flatten the arrays into continuous timelines
        all_preds_game = np.concatenate(all_preds_game, axis=0)
        all_probs_game = np.concatenate(all_probs_game, axis=0)
        all_states_game = np.concatenate(all_states_game, axis=0)

        # 3. FEED THE SIMULATOR
        # This prevents the simulator from hallucinating execution without accounting for queue position[cite: 101, 102].
        # It runs your exact queue-depletion and TTL logic for this specific game.
        game_pnl, game_trades = simulator_instance.run_market(
            predictions=all_preds_game,
            probabilities=all_probs_game,
            states=all_states_game,
            threshold=trigger_threshold
        )

        total_pnl += game_pnl
        total_trades += game_trades

    print("\n=== FINAL SIMULATOR RESULTS ===")
    print(f"Total Trades Executed: {total_trades}")
    print(f"Total Net PnL: {total_pnl:.2f}¢")
    return total_pnl

# Usage Example:
#my_simulator = KalshiBacktestSimulator(latency_penalty_ms=50, ttl_seconds=15)
#evaluate_model_with_simulator(model, val_files, my_simulator, trigger_threshold=0.6)

--- Booting Simulator Harness on 10 Validation Markets ---
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 311 | FLAT: 13306 | UP: 693
Dropped due to NaNs: 0


TypeError: KalshiBacktestSimulator.run_market() missing 2 required positional arguments: 'timestamps' and 'raw_market_df'

In [23]:
import glob
import pandas as pd
import os

def load_raw_market_df(ticker: str, raw_extracted_base_dir: str = './kalshi_extracted') -> pd.DataFrame:
    """Finds all raw parquet partitions for a specific market and builds the tick-by-tick dataframe."""
    # Find the specific folder for this ticker in the uncompressed raw data
    search_path = f"{raw_extracted_base_dir}/**/{ticker}/*.parquet"
    partition_files = glob.glob(search_path, recursive=True)

    if not partition_files:
        raise FileNotFoundError(f"Could not find raw partitions for {ticker}.")

    # Concatenate and sort exactly like we did in the offline script
    df = pd.concat([pd.read_parquet(f) for f in partition_files], ignore_index=True)
    df = df.sort_values(by=['local_recv_ts', 'seq']).reset_index(drop=True)

    return df

# === Example Usage in your Evaluation Loop ===

# Let's say we are evaluating one of your validation files
"""dense_val_file = 'dense_KXMLBGAME-26APR161240SFCIN-SF.parquet'

# 1. Extract the raw ticker name from your dense file name
ticker_name = dense_val_file.replace('dense_', '').replace('.parquet', '')

# 2. Load the RAW, microsecond-level tick data for the simulator
raw_df = load_raw_market_df(ticker_name)

# 3. Call the harness
my_sim = KalshiBacktestSimulator(latency_penalty_ms=50, ttl_seconds=15)
evaluate_model_with_simulator(
    alpha_model=model,
    val_file_paths=[dense_val_file],
    simulator_instance=my_sim,
    raw_market_df=raw_df,  # <--- Pass it here
    trigger_threshold=0.6
)"""



In [24]:
import glob

# Search the entire Colab environment for ANY parquet file belonging to a specific validation ticker
# Replace this ticker with whatever validation game you are testing
test_ticker = "KXMLBGAME-26APR161240SFCIN-SF"

print(f"Searching for raw data for: {test_ticker}...")
found_files = glob.glob(f"/content/**/{test_ticker}/*.parquet", recursive=True)

if found_files:
    print(f"SUCCESS! Found {len(found_files)} raw partition files.")
    print(f"They are located at: {found_files[0]}")
else:
    print("FAILED. Could not find any raw parquet files for this ticker.")
    print("Did you extract the .tar archives in Colab? You must run: !tar -xf raw_vault_2026-04-16.tar")

Searching for raw data for: KXMLBGAME-26APR161240SFCIN-SF...
FAILED. Could not find any raw parquet files for this ticker.
Did you extract the .tar archives in Colab? You must run: !tar -xf raw_vault_2026-04-16.tar


In [25]:
# Unpack the tar archives into the Colab /content/ directory
!tar -xf raw_vault_2026-04-15.tar -C /content/
!tar -xf raw_vault_2026-04-16.tar -C /content/

print("Extraction complete. Raw tick data is now available to the simulator.")

Extraction complete. Raw tick data is now available to the simulator.


In [26]:
import glob
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def load_raw_market_df(ticker: str, raw_base_dir: str = '/content/') -> pd.DataFrame:
    """Automatically hunts down the raw tick data for a given market."""
    search_path = f"{raw_base_dir}/**/{ticker}/*.parquet"
    partition_files = glob.glob(search_path, recursive=True)

    if not partition_files:
        raise FileNotFoundError(f"CRITICAL: Could not find raw tick data for {ticker}. Did the .tar extract correctly?")

    df = pd.concat([pd.read_parquet(f) for f in partition_files], ignore_index=True)
    df = df.sort_values(by=['local_recv_ts', 'seq']).reset_index(drop=True)
    return df

def run_full_validation_suite(alpha_model, val_dense_files: list, trigger_threshold: float = 0.75):
    """
    The ONE-CLICK harness.
    It automatically pairs the dense ML tensors with the raw execution data.
    """
    alpha_model.eval()
    master_sim = KalshiBacktestSimulator(latency_penalty_ms=50, ttl_seconds=15)

    print(f"=== BOOTING VALIDATION SUITE ACROSS {len(val_dense_files)} MARKETS ===")

    total_pnl = 0.0
    total_trades = 0

    for dense_file in val_dense_files:
        # 1. Automatically extract the ticker from the file name
        ticker_name = os.path.basename(dense_file).replace('dense_', '').replace('.parquet', '')

        try:
            # 2. Automatically load the corresponding RAW microsecond data
            raw_df = load_raw_market_df(ticker_name)

            # 3. Load the corresponding DENSE 1-second ML data
            market_dataset = KalshiDenseDataset([dense_file], seq_len=60, forward_look=30, threshold=0.015, stride=1)
            if len(market_dataset) == 0:
                continue

            market_loader = DataLoader(market_dataset, batch_size=256, shuffle=False)

            all_preds, all_probs, all_states, all_ts = [], [], [], []

            # 4. Generate model predictions chronologically
            with torch.no_grad():
                for X_batch, Y_batch, timestamps in market_loader:
                    X_batch = X_batch.to(device)
                    logits = alpha_model(X_batch)
                    probs = F.softmax(logits, dim=1)

                    all_preds.append(torch.argmax(probs, dim=1).cpu().numpy())
                    all_probs.append(probs.cpu().numpy())
                    all_states.append(X_batch.cpu().numpy())
                    all_ts.append(timestamps.numpy())

            # 5. Feed the paired data directly into the execution simulator
            game_pnl, game_trades = master_sim.run_market(
                predictions=np.concatenate(all_preds, axis=0),
                probabilities=np.concatenate(all_probs, axis=0),
                states=np.concatenate(all_states, axis=0),
                timestamps=np.concatenate(all_ts, axis=0),
                raw_market_df=raw_df,
                threshold=trigger_threshold
            )

            total_pnl += game_pnl
            total_trades += game_trades

        except Exception as e:
            print(f"Skipping {ticker_name} due to error: {str(e)}")

    print("\n=== FINAL VALIDATION PNL ===")
    master_sim.print_autopsy()
    return total_pnl

In [37]:
# 1. Define your confidence threshold (e.g., from your previous autopsy)
CONFIDENCE_TRIGGER = 0.6

# 2. Execute the one-click harness
total_validation_pnl = run_full_validation_suite(
    alpha_model=model,
    val_dense_files=val_files,
    trigger_threshold=CONFIDENCE_TRIGGER
)

=== BOOTING VALIDATION SUITE ACROSS 10 MARKETS ===
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 311 | FLAT: 13306 | UP: 693
Dropped due to NaNs: 0
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 661 | FLAT: 13269 | UP: 380
Dropped due to NaNs: 0
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 853 | FLAT: 12784 | UP: 673
Dropped due to NaNs: 0
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 652 | FLAT: 12800 | UP: 858
Dropped due to NaNs: 0
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 14310
Distribution -> DOWN: 1009 | FLAT: 12652 | UP: 649
Dropped due to NaNs: 0
Loading 1 dense files into RAM...

=== FINAL ENGINEERING AUDIT 